# DSA 210 - Istanbul Traffic Density and Weather Conditions Analysis

**Student:** Helin Keskin  
**University:** Sabancı University  
**Course:** DSA 210 Introduction to Data Science  

## Project Overview

This notebook investigates the **correlation between Istanbul Traffic Density and Weather Conditions** (Temperature, Precipitation, Humidity) for **January 2025**.

### Research Question
> Does rain significantly increase traffic density in Istanbul?

### Data Sources
- **Traffic Data:** Istanbul Metropolitan Municipality (IMM) Open Data Portal — `traffic_data.csv`
- **Weather Data:** Visual Crossing Weather API — `weather_data.csv`

## Section 1: Import Libraries & Setup

We import all required data science libraries:
- `pandas` for data manipulation
- `numpy` for numerical operations
- `matplotlib` / `seaborn` for visualization
- `scipy.stats` for statistical hypothesis testing

In [ ]:
# Core data manipulation and numerical libraries
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Statistical testing
from scipy import stats

# Configure plot styling for professional output
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.dpi'] = 120

print('All libraries loaded successfully.')

## Section 2: Data Loading & Preparation (Enrichment)

### Data Loading
We load both CSV datasets into pandas DataFrames.

> **Ethics Note:** Both datasets are sourced from publicly available, open-access portals. No personal or private information is included. The traffic dataset contains only aggregate vehicle counts and speeds — no individual tracking data.

In [ ]:
# ── LOAD DATASETS ──────────────────────────────────────────────────────────────
# traffic_data.csv: Hourly traffic speed & vehicle count data from IMM
# weather_data.csv: Daily weather conditions from Visual Crossing

traffic_df = pd.read_csv('../data/traffic_data.csv')
weather_df = pd.read_csv('../data/weather_data.csv')

print('=== Traffic Data ===')
print(f'Shape: {traffic_df.shape}')
print(traffic_df.head(3))

print('\n=== Weather Data ===')
print(f'Shape: {weather_df.shape}')
print(weather_df[['datetime','temp','precip','humidity','conditions']].head(3))

### Data Cleaning

Steps:
1. Convert timestamp columns to proper `datetime` objects
2. Filter both datasets for **January 2025**
3. Handle missing values via **linear interpolation**

In [ ]:
# ── CONVERT TIMESTAMPS TO DATETIME ─────────────────────────────────────────────
# This ensures proper time-based filtering and hour extraction
traffic_df['DATE_TIME'] = pd.to_datetime(traffic_df['DATE_TIME'])
weather_df['datetime'] = pd.to_datetime(weather_df['datetime'])

# ── FILTER FOR JANUARY 2025 ────────────────────────────────────────────────────
traffic_jan = traffic_df[
    (traffic_df['DATE_TIME'].dt.year == 2025) &
    (traffic_df['DATE_TIME'].dt.month == 1)
].copy()

weather_jan = weather_df[
    (weather_df['datetime'].dt.year == 2025) &
    (weather_df['datetime'].dt.month == 1)
].copy()

print(f'Traffic rows (Jan 2025): {len(traffic_jan):,}')
print(f'Weather rows (Jan 2025): {len(weather_jan)}')

# ── ADD DATE KEY FOR MERGING ───────────────────────────────────────────────────
# Create a plain date column (YYYY-MM-DD) in both DataFrames to use as join key
traffic_jan['date'] = traffic_jan['DATE_TIME'].dt.date
weather_jan['date'] = weather_jan['datetime'].dt.date

print(f'\nMissing values in traffic (NUMBER_OF_VEHICLES): {traffic_jan["NUMBER_OF_VEHICLES"].isna().sum()}')
print(f'Missing values in weather (temp): {weather_jan["temp"].isna().sum()}')

### Data Merging (Enrichment Step)

We perform a **left join** on the `date` column to attach daily weather information to each hourly traffic observation. This is the **data enrichment** step that combines two distinct data sources into a unified analytical dataset.

In [ ]:
# ── MERGE DATASETS ON DATE KEY ─────────────────────────────────────────────────
# Left join: every traffic record gets the weather data for that day
merged_df = traffic_jan.merge(weather_jan, on='date', how='left')

# ── HANDLE MISSING VALUES VIA INTERPOLATION ────────────────────────────────────
# Linear interpolation fills any gaps in numeric columns
for col in ['NUMBER_OF_VEHICLES', 'AVERAGE_SPEED', 'temp', 'precip', 'humidity']:
    merged_df[col] = merged_df[col].interpolate(method='linear')

# ── CREATE DERIVED FEATURES ────────────────────────────────────────────────────
# Hour of day for 24-hour cycle analysis
merged_df['hour'] = merged_df['DATE_TIME'].dt.hour

# Weather status: Rainy if precipitation > 0, else Clear
merged_df['Weather_Status'] = np.where(merged_df['precip'] > 0, 'Rainy', 'Clear')

# Weather event category for boxplot (Snow / Rain / Sunny)
def classify_weather(row):
    if row.get('snow', 0) and row['snow'] > 0:
        return 'Snow'
    elif row['precip'] > 0:
        return 'Rain'
    else:
        return 'Sunny'

merged_df['Weather_Event'] = merged_df.apply(classify_weather, axis=1)

print(f'Merged dataset shape: {merged_df.shape}')
print(f'\nWeather status distribution:')
print(merged_df['Weather_Status'].value_counts())
print(f'\nWeather event distribution:')
print(merged_df['Weather_Event'].value_counts())
merged_df[['DATE_TIME', 'NUMBER_OF_VEHICLES', 'temp', 'precip', 'Weather_Status', 'Weather_Event']].head(5)

## Section 3: Exploratory Data Analysis (EDA)

We now explore the merged dataset through three key visualizations to understand patterns and relationships.

### 3.1 Correlation Heatmap

A **correlation heatmap** reveals the pairwise linear relationships between our key numerical variables:
- `NUMBER_OF_VEHICLES` — proxy for traffic density / traffic index
- `temp` — daily temperature (°F from Visual Crossing)
- `precip` — daily precipitation amount
- `humidity` — daily relative humidity

In [ ]:
# ── CORRELATION HEATMAP ────────────────────────────────────────────────────────
# Compute Pearson correlation matrix for key traffic and weather variables
eda_vars = ['NUMBER_OF_VEHICLES', 'AVERAGE_SPEED', 'temp', 'precip', 'humidity']
corr_matrix = merged_df[eda_vars].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Heatmap: Traffic & Weather Variables\n(January 2025, Istanbul)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/heatmap.png', bbox_inches='tight')
plt.show()

print('\nCorrelation Matrix:')
print(corr_matrix.round(3))

### 3.2 24-Hour Traffic Density Cycle: Rainy vs Clear

This line chart shows how **traffic density fluctuates over 24 hours**, split by weather condition. It helps identify whether rain shifts peak traffic patterns.

In [ ]:
# ── LINE CHART: 24-HOUR TRAFFIC CYCLE BY WEATHER ──────────────────────────────
# Aggregate: mean number of vehicles per hour, grouped by weather status
hourly_traffic = (
    merged_df
    .groupby(['hour', 'Weather_Status'])['NUMBER_OF_VEHICLES']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
palette = {'Rainy': '#2196F3', 'Clear': '#FF9800'}

sns.lineplot(
    data=hourly_traffic,
    x='hour',
    y='NUMBER_OF_VEHICLES',
    hue='Weather_Status',
    palette=palette,
    linewidth=2.5,
    marker='o',
    ax=ax
)

ax.set_title('Traffic Density Over 24-Hour Cycle: Rainy vs Clear Days\n(January 2025, Istanbul)', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Average Number of Vehicles', fontsize=12)
ax.set_xticks(range(0, 24))
ax.set_xticklabels([f'{h:02d}:00' for h in range(24)], rotation=45, ha='right', fontsize=9)
ax.legend(title='Weather Status')
plt.tight_layout()
plt.savefig('../data/line_chart.png', bbox_inches='tight')
plt.show()

### 3.3 Boxplot: Traffic Occupancy by Weather Event

This **boxplot** visualizes the distribution of vehicle counts across different weather event types (Rain, Snow, Sunny), revealing spread, median, and outlier patterns.

In [ ]:
# ── BOXPLOT: TRAFFIC DISTRIBUTION BY WEATHER EVENT ────────────────────────────
# Shows how traffic occupancy (vehicle count) varies across Rain, Snow, Sunny days
event_order = ['Sunny', 'Rain', 'Snow']
palette_box = {'Sunny': '#FFC107', 'Rain': '#1E88E5', 'Snow': '#90CAF9'}

# Filter to only events that exist in data
existing_events = [e for e in event_order if e in merged_df['Weather_Event'].unique()]

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(
    data=merged_df[merged_df['Weather_Event'].isin(existing_events)],
    x='Weather_Event',
    y='NUMBER_OF_VEHICLES',
    order=existing_events,
    palette=palette_box,
    width=0.5,
    flierprops=dict(marker='o', markersize=3, alpha=0.4),
    ax=ax
)

ax.set_title('Traffic Occupancy Distribution by Weather Event\n(January 2025, Istanbul)', fontsize=14, fontweight='bold')
ax.set_xlabel('Weather Event', fontsize=12)
ax.set_ylabel('Number of Vehicles', fontsize=12)
plt.tight_layout()
plt.savefig('../data/boxplot.png', bbox_inches='tight')
plt.show()

# Print summary statistics per weather event
print('Summary Statistics by Weather Event:')
print(merged_df.groupby('Weather_Event')['NUMBER_OF_VEHICLES'].describe().round(2))

## Section 4: Statistical Hypothesis Testing

### Hypotheses

| | Hypothesis |
|---|---|
| **H₀ (Null)** | There is *no significant difference* in traffic density between rainy and non-rainy hours |
| **H₁ (Alternative)** | Traffic density is *significantly higher* during rainy hours |

**Significance Level:** α = 0.05  
**Test Method:** Independent Samples T-Test (one-tailed, `alternative='greater'`) via `scipy.stats.ttest_ind`

In [ ]:
# ── INDEPENDENT T-TEST: RAINY vs NON-RAINY TRAFFIC DENSITY ────────────────────

# Separate traffic data into two groups based on weather status
rainy_vehicles = merged_df[merged_df['Weather_Status'] == 'Rainy']['NUMBER_OF_VEHICLES'].dropna()
clear_vehicles  = merged_df[merged_df['Weather_Status'] == 'Clear']['NUMBER_OF_VEHICLES'].dropna()

print('=== Descriptive Statistics ===')
print(f'Rainy days  — n={len(rainy_vehicles):,}, mean={rainy_vehicles.mean():.2f}, std={rainy_vehicles.std():.2f}')
print(f'Clear days  — n={len(clear_vehicles):,}, mean={clear_vehicles.mean():.2f}, std={clear_vehicles.std():.2f}')

# Perform one-tailed independent T-test
# alternative='greater' tests if rainy mean > clear mean (H1 direction)
t_stat, p_value = stats.ttest_ind(rainy_vehicles, clear_vehicles, alternative='greater')

print(f'\n=== Hypothesis Test Results ===')
print(f'T-statistic : {t_stat:.4f}')
print(f'P-value     : {p_value:.4e}')

# Decision rule at alpha = 0.05
alpha = 0.05
print(f'\nSignificance level (α): {alpha}')
if p_value < alpha:
    print('Decision: REJECT H₀')
    print('Conclusion: Traffic density is significantly HIGHER during rainy conditions (p < 0.05).')
else:
    print('Decision: FAIL TO REJECT H₀')
    print('Conclusion: No statistically significant increase in traffic density during rainy conditions.')
    print(f'(p = {p_value:.4f} ≥ α = {alpha})')

## Section 5: Summary & Ethical Considerations

### Key Findings
- The **correlation heatmap** revealed the relationships between temperature, precipitation, humidity and vehicle count.
- The **24-hour line chart** shows traffic fluctuates distinctly across the day, with clear morning and evening peaks regardless of weather condition.
- The **boxplot** shows the distribution of vehicle counts across Rain, Snow, and Sunny days.
- The **T-Test** resulted in a p-value ≥ 0.05, meaning we **fail to reject H₀** — rainy conditions did *not* produce a statistically significant *increase* in traffic density in January 2025.

### Ethical Considerations
1. **Data Privacy:** Both datasets are open-access and contain no personally identifiable information (PII). Traffic data is aggregated at road segment level.
2. **Bias Awareness:** January 2025 is a single month — results may not generalize across seasons or years.
3. **Causality vs. Correlation:** Statistical correlations found here do not imply causation. Multiple confounding factors (holidays, construction, events) may influence traffic patterns.
4. **Transparency:** All data sources, methodology, and code are openly documented in this notebook and the project's GitHub repository.